# video-app lab 10 - STEP CHANGE: Wan2.2 TI2V-5B (2025 model, Apache-2.0)
Why: lab 09 used AnimateDiff + SD1.5 = 2023 technology, distilled to 8 steps. Soft, washed-out, weak scene
understanding is its ceiling, not a bug. Wan2.2 5B is a current open model: real prompt understanding,
720p24 native, Apache-2.0, free weights.

Runtime needed: **L4 GPU (24 GB) or A100** - T4 cannot hold it. L4 burns more Colab compute units than T4.
Expect ~10-20 min per clip. We start small (480x832, 49 frames) and step up only if it fits.


In [ ]:
import os, shutil, psutil, torch
vm = psutil.virtual_memory()
print(f'RAM {vm.used/1024**3:.1f}/{vm.total/1024**3:.1f} GB | disk {shutil.disk_usage("/").free/1024**3:.0f} GB | cuda {torch.cuda.is_available()}')
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU -> Runtime > Change runtime type > L4 GPU (or A100) + High-RAM > Save')
p = torch.cuda.get_device_properties(0)
print('GPU', p.name, '| VRAM', round(p.total_memory/1024**3,1), 'GB')
if p.total_memory < 20*1024**3: raise SystemExit('STOP: this GPU has <20 GB VRAM. Wan2.2 5B needs L4 (24 GB) or A100 (40 GB). Runtime > Change runtime type.')
print('bf16:', torch.cuda.is_bf16_supported())


In [ ]:
import gc, time, torch
from diffusers import AutoencoderKLWan, WanPipeline
REPO = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
t0=time.time()
vae = AutoencoderKLWan.from_pretrained(REPO, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(REPO, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
try: pipe.vae.enable_tiling()
except Exception as e: print('vae tiling skip', type(e).__name__)
print('loaded in', round(time.time()-t0), 's | VRAM', round(torch.cuda.memory_allocated()/1024**3,2), 'GB')


In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
PROMPT = ('Wide cinematic shot of a red fox trotting through deep snow between tall snow-covered pine trees, '
          'low golden winter sunbeams cutting through the branches, snow dust kicking up from its paws, '
          'fur detail, shallow depth of field, steady tracking camera, photorealistic, 35mm film')
NEG = 'blurry, low quality, distorted, extra limbs, text, watermark, flicker'
def run(h, w, frames, steps):
    t0=time.time()
    out = pipe(prompt=PROMPT, negative_prompt=NEG, height=h, width=w, num_frames=frames,
               num_inference_steps=steps, guidance_scale=5.0,
               generator=torch.Generator('cpu').manual_seed(11)).frames[0]
    print(f'OK {w}x{h} {frames}f {steps} steps in {round(time.time()-t0)} s')
    return out
try:
    frames = run(480, 832, 49, 20)
except torch.cuda.OutOfMemoryError:
    print('OOM -> retry smaller'); torch.cuda.empty_cache(); gc.collect()
    frames = run(384, 672, 33, 16)
path='/content/clip_10_wan22.mp4'
export_to_video(frames, path, fps=24)
print('MB', round(os.path.getsize(path)/1024**2,1))
display(Video(path, embed=True, width=640))
from google.colab import files; files.download(path)


### Scaling after it works
720p = 704x1280, 121 frames (5 s at 24 fps), 30 steps: the model's native quality. Only on A100 40 GB.
The point of this run is proof of quality, not final size.
